# Fine-tune an ~8B LLM with full / LoRA / QLoRA

Parts 1 and 2 use small models, so QLoRA barely changes the memory picture. An **8B** model is where the three methods diverge — and where one **RTX PRO 6000 Blackwell (96 GB)** can still do **full** fine-tuning on one card.

Same text-to-SQL task and Flyte 2 pipeline as [part 1](../01_text_to_sql/). Default checkpoint: [`meta-llama/Llama-3.1-8B`](https://huggingface.co/meta-llama/Llama-3.1-8B). Ungated fallback: [`mistralai/Mistral-7B-v0.3`](https://huggingface.co/mistralai/Mistral-7B-v0.3).

**Reported runs** use **NVIDIA RTX PRO 6000 Blackwell (96 GB)** — one card or four on one node (16-core, 256 GB host). Follow along with LoRA or QLoRA on a smaller NVIDIA GPU; full 8B fine-tuning is the RTX PRO 6000 Blackwell run.

## Why this size

| Piece (8B, bf16, AdamW) | ~Memory |
|-------------------------|---------|
| Weights | 16 GB |
| Gradients | 16 GB |
| AdamW (m + v in fp32) | 64 GB |
| **Before activations** | **~96 GB** |

| Method | Typical VRAM | Who can run it |
|--------|--------------|----------------|
| Full | ~96 GB + activations (checkpointing, batch size 1) | NVIDIA RTX PRO 6000 Blackwell (96 GB) |
| LoRA | ~20–24 GB | Most recent NVIDIA GPUs |
| QLoRA | ~6–12 GB | 16–24 GB cards; the “you don’t need 96 GB” path |

Use the **base** model, not Instruct, so the before/after SQL eval stays as readable as SmolLM2.

## Models

| Hugging Face id | Size | Token? |
|-----------------|------|--------|
| `meta-llama/Llama-3.1-8B` | 8B | Yes (accept the license, `HF_TOKEN`) |
| `mistralai/Mistral-7B-v0.3` | 7B | No |
| `allenai/OLMo-2-1124-7B` | 7B | No |
| `google/gemma-2-9b` | 9B | Check the model card |

LoRA `target_modules` are the same LLaMA-style names already in `../01_text_to_sql/workflow.py`.

In [ ]:
MODEL = "meta-llama/Llama-3.1-8B"
# MODEL = "mistralai/Mistral-7B-v0.3"  # ungated

print("From NLP/Fine_Tuning/01_text_to_sql/ run:")
for method in ("lora", "qlora", "full"):
    print(
        f"flyte run --local --tui workflow.py pipeline "
        f"--model_name {MODEL} --method {method} --epochs 1 "
        f"--max-train-samples 100 --max-eval-samples 20 --batch_size 1"
    )

## Run order

1. Smoke test LoRA with 100 train rows (downloads the 8B weights once; they stay in the Hugging Face cache, which is gitignored).
2. QLoRA on the same subset.
3. Full fine-tune on the RTX PRO 6000 Blackwell only. If it OOMs, turn on gradient checkpointing in `workflow.py` before a longer run.
4. Scale `--max-train-samples` toward the part 1 default (5000) once the three methods complete.

Fill the comparison table in the part 3 README after the RTX PRO 6000 Blackwell runs: trainable params, peak VRAM, wall time, SQL exact match — next to the SmolLM2-135M numbers from part 1.